In [2]:
from secedgar.cik_lookup import get_cik_map
import requests
import pandas as pd
import time
import io

USER_AGENT = "Academic Research Project (gig-economy-capstone@umich.edu)"

c:\Python311\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.4.0) or chardet (6.0.0.post1)/charset_normalizer (3.4.2) doesn't match a supported version!
  warnings.warn(


In [3]:
upwk_cik = '1627475'
cik_padded = str(upwk_cik).zfill(10)

cik_padded = upwk_cik.zfill(10)
url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik_padded}.json"

headers = {
    'User-Agent': USER_AGENT,
    'Accept-Encoding': 'gzip, deflate'
}

response = requests.get(url, headers=headers, timeout=30)
response.raise_for_status()
data = response.json()

results = []
facts = data.get('facts', {})

# US-GAAP namespace contains most financial data
us_gaap = facts.get('us-gaap', {})
# all available metrics in the us-gaap namespace
available_metrics = list(us_gaap.keys())
print(f"Available metrics for CIK {cik_padded}: {available_metrics}")

# all available units
available_units = set()
for metric, metric_data in us_gaap.items():
    for unit in metric_data.get('units', {}):
        available_units.add(unit)
print(f"Available units for CIK {cik_padded}: {available_units}")

# list all metrics that include 'Outstanding' in their name
# outstanding_metrics = [metric for metric in available_metrics if 'Outstanding' in metric]
# print(f"Outstanding-related metrics for CIK {cik_padded}: {outstanding_metrics}")

Available metrics for CIK 0001627475: ['AccountsPayableCurrent', 'AccountsReceivableGrossCurrent', 'AccountsReceivableNetCurrent', 'AccretionAmortizationOfDiscountsAndPremiumsInvestments', 'AccumulatedDepreciationDepletionAndAmortizationPropertyPlantAndEquipment', 'AdditionalPaidInCapitalCommonStock', 'AdjustmentsToAdditionalPaidInCapitalEquityComponentOfConvertibleDebtSubsequentAdjustments', 'AdjustmentsToAdditionalPaidInCapitalSharebasedCompensationRequisiteServicePeriodRecognitionValue', 'AdjustmentsToAdditionalPaidInCapitalStockIssuedIssuanceCosts', 'AdjustmentsToAdditionalPaidInCapitalWarrantIssued', 'AdvertisingExpense', 'AllocatedShareBasedCompensationExpense', 'AllowanceForDoubtfulAccountsReceivable', 'AllowanceForDoubtfulAccountsReceivableCurrent', 'AllowanceForDoubtfulAccountsReceivableWriteOffs', 'AmortizationOfFinancingCosts', 'AmortizationOfIntangibleAssets', 'AntidilutiveSecuritiesExcludedFromComputationOfEarningsPerShareAmount', 'AssetImpairmentCharges', 'Assets', 'Asset

In [4]:
concepts_to_fetch = ['Revenues', 'GrossProfit', 'OperatingIncomeLoss', 'NetIncomeLoss', 'OperatingExpenses',
                     'Assets', 'Liabilities', 'RevenueFromContractWithCustomerExcludingAssessedTax', 
                     'IncomeTaxExpenseBenefit', 'CommonStockSharesOutstanding', 
                     'WeightedAverageNumberOfSharesOutstandingBasic']

In [5]:
def fetch_tickers(url="", table=0, index_col="Symbol", company_col="Security",  cik_col="CIK"):
    # Fetch with proper User-Agent header to avoid 403 error
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
    response = requests.get(url, headers=headers)
    tables = pd.read_html(io.StringIO(response.text))
    index_table = tables[table]  # Use the specified table index
    tickers = index_table[index_col].tolist()
    companies = index_table[company_col].tolist() if company_col in index_table.columns else None
    # cik = index_table[cik_col].tolist() if cik_col in index_table.columns else Nones
    
    # dataframe of tickers, companies, cik
    df = pd.DataFrame({
        'Company': companies,
        'Ticker': tickers,
        # 'CIK': cik
    })

    return df

# nasdaq_100_url = 'https://en.wikipedia.org/wiki/NASDAQ-100'
sp500_url = 'http://en.wikipedia.org/wiki/List_of_S%26P_500_companies'

In [6]:
sp500_df = fetch_tickers(sp500_url)
# convert sp500_df to dictionary with company as key and ticker as value
sp500_tickers = dict(zip(sp500_df['Company'], sp500_df['Ticker']))
sp500_tickers

{'3M': 'MMM',
 'A. O. Smith': 'AOS',
 'Abbott Laboratories': 'ABT',
 'AbbVie': 'ABBV',
 'Accenture': 'ACN',
 'Adobe Inc.': 'ADBE',
 'Advanced Micro Devices': 'AMD',
 'AES Corporation': 'AES',
 'Aflac': 'AFL',
 'Agilent Technologies': 'A',
 'Air Products': 'APD',
 'Airbnb': 'ABNB',
 'Akamai Technologies': 'AKAM',
 'Albemarle Corporation': 'ALB',
 'Alexandria Real Estate Equities': 'ARE',
 'Align Technology': 'ALGN',
 'Allegion': 'ALLE',
 'Alliant Energy': 'LNT',
 'Allstate': 'ALL',
 'Alphabet Inc. (Class A)': 'GOOGL',
 'Alphabet Inc. (Class C)': 'GOOG',
 'Altria': 'MO',
 'Amazon': 'AMZN',
 'Amcor': 'AMCR',
 'Ameren': 'AEE',
 'American Electric Power': 'AEP',
 'American Express': 'AXP',
 'American International Group': 'AIG',
 'American Tower': 'AMT',
 'American Water Works': 'AWK',
 'Ameriprise Financial': 'AMP',
 'Ametek': 'AME',
 'Amgen': 'AMGN',
 'Amphenol': 'APH',
 'Analog Devices': 'ADI',
 'Aon plc': 'AON',
 'APA Corporation': 'APA',
 'Apollo Global Management': 'APO',
 'Apple Inc.

In [7]:
gig_tickers = {
    'Upwork': "UPWK",
    'Fiverr': "FVRR",
    'Uber': "UBER",
    'Lyft': "LYFT",
    'Doordash': "DASH",
    'Grubhub': "GRUB",
    'Instacart': "CART",
    'Etsy': "ETSY",
    'Shopify': "SHOP",
    'Udemy': "UDMY",
    'Herbalife': "HLF",
    'Primerica': "PRI",
    'Tupperware': "TUP",
    'Avon': "AVP",
}

# Get the CIK map - returns dict with "ticker" and "title" keys
cik_map = get_cik_map(user_agent='Academic Research Project (gig-economy-capstone@umich.edu)')

# Access ticker -> CIK mapping (keys are uppercase)
ticker_to_cik = cik_map["ticker"]

# Debug: check what keys look like
# sample_tickers = list(ticker_to_cik.keys())[:10]
# print(f"Sample tickers in CIK map: {sample_tickers}")

# Lookup CIKs for our tickers
gig_ciks = {}
for company, ticker in gig_tickers.items():
    lookup = ticker.upper()
    if lookup in ticker_to_cik:
        gig_ciks[ticker] = ticker_to_cik[lookup]
    else:
        print(f"Ticker {ticker} not found in CIK map")

print(f"\nFound CIKs: {gig_ciks}")

sp500_ciks = {}
for company, ticker in sp500_tickers.items():
    lookup = ticker.upper()
    if lookup in ticker_to_cik:
        sp500_ciks[ticker] = ticker_to_cik[lookup]
    else:
        print(f"Ticker {ticker} not found in CIK map")

print(f"\nFound CIKs for S&P 500: {len(sp500_ciks)} companies found")

Ticker GRUB not found in CIK map
Ticker TUP not found in CIK map
Ticker AVP not found in CIK map

Found CIKs: {'UPWK': '1627475', 'FVRR': '1762301', 'UBER': '1543151', 'LYFT': '1759509', 'DASH': '1792789', 'CART': '1579091', 'ETSY': '1370637', 'SHOP': '1594805', 'UDMY': '1607939', 'HLF': '1180262', 'PRI': '1475922'}
Ticker BRK.B not found in CIK map
Ticker BF.B not found in CIK map

Found CIKs for S&P 500: 501 companies found


In [8]:
# recovered missing gig company CIKs by looking up tickers manually
# add missing gig company CIKs to gig_ciks dictionary
gig_ciks['GRUB'] = '0001594109'
gig_ciks['TUP'] = '0001008654'
gig_ciks['AVON'] = '0000008868'
gig_ciks

{'UPWK': '1627475',
 'FVRR': '1762301',
 'UBER': '1543151',
 'LYFT': '1759509',
 'DASH': '1792789',
 'CART': '1579091',
 'ETSY': '1370637',
 'SHOP': '1594805',
 'UDMY': '1607939',
 'HLF': '1180262',
 'PRI': '1475922',
 'GRUB': '0001594109',
 'TUP': '0001008654',
 'AVON': '0000008868'}

In [9]:
# to a dictionary with ticker as key and dictionary of ticker and cik
gig_ciks_dict = {ticker: {"ticker": ticker, "cik": str(cik)} for ticker, cik in gig_ciks.items()}
sp500_ciks_dict = {ticker: {"ticker": ticker, "cik": str(cik)} for ticker, cik in sp500_ciks.items()}

In [10]:
def get_company_financials_xbrl(cik, metrics=None):
    if metrics is None:
        metrics = concepts_to_fetch
    
    cik_padded = cik.zfill(10)
    url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik_padded}.json"
    
    headers = {
        'User-Agent': 'Academic Research Project (gig-economy-capstone@umich.edu)',
        'Accept-Encoding': 'gzip, deflate'
    }
    
    try:
        response = requests.get(url, headers=headers, timeout=30)
        response.raise_for_status()
        data = response.json()
        
        results = []
        facts = data.get('facts', {})
        
        # US-GAAP namespace contains most financial data
        us_gaap = facts.get('us-gaap', {})
        units = {'Segment', 'shares', 'segment', 'tradingDay', 'pure', 'USD/shares', 'USD'}
        data_items = []
        for metric in metrics:
            if metric in us_gaap:
                for unit in units:
                    if unit in us_gaap[metric].get('units', {}):
                        data_items.extend(us_gaap[metric]['units'][unit])      
                for item in data_items:
                    if item.get('form') in ['10-K', '10-Q', '20-F', '6-K']:
                        results.append({
                            'form': item.get('form'),
                            'metric': metric,
                            'value': item.get('val'),
                            'fiscal_year': item.get('fy'),
                            'fiscal_period': item.get('fp'),
                            'filed_date': item.get('filed'),
                            'start_date': item.get('start'),
                            'end_date': item.get('end'),
                            # 'accession_number': item.get('accn'),
                        })
        
        df = pd.DataFrame(results)
        
        if not df.empty:
            # Pivot to get metrics as columns
            df_pivot = df.pivot_table(
                index=['filed_date', 'fiscal_year', 'form', 'fiscal_period'],
                columns='metric',
                values='value',
                aggfunc='first'
            ).reset_index()
            
            # add company and ticker as first columns
            
            return df_pivot
        
        return df
        
    except requests.exceptions.RequestException as e:
        print(f"Error fetching XBRL data for CIK {cik}: {e}")
        return pd.DataFrame()


def get_all_companies_financials(companies=None):
    if companies is None:
        companies = gig_ciks_dict
    
    all_financials = []
    
    for company_name, company_info in companies.items():
        print(f"Fetching financials for {company_name}...")
        time.sleep(0.1)  # Rate limiting
        
        try:
            df = get_company_financials_xbrl(company_info['cik'])
            if not df.empty:
                df['company'] = company_name
                df['ticker'] = company_info['ticker']
                # df['category'] = company_info['category']
                all_financials.append(df)
                print(f"  Found {len(df)} annual records")
            else:
                print(f"  No financial data found")
        except Exception as e:
            print(f"  Error: {e}")
            continue
    
    if all_financials:
        combined = pd.concat(all_financials, ignore_index=True)
        combined = combined.rename_axis(None, axis=1)
        return combined
    else:
        return pd.DataFrame()

In [26]:
# gig_edgar_df_2 = get_all_companies_financials(gig_ciks_dict)
# gig_edgar_df_2 = gig_edgar_df_2[gig_edgar_df_2['fiscal_year'] != 2026].reset_index(drop=True)
# gig_edgar_df_2.to_csv("../../data/edgar_sec/gig_edgar_sec_raw.csv", index=False)
gig_edgar_df_2 = pd.read_csv("../../data/edgar_sec/gig_edgar_sec_raw.csv")
gig_edgar_df_2

,filed_date,fiscal_year,form,fiscal_period,Assets,CommonStockSharesOutstanding,GrossProfit,IncomeTaxExpenseBenefit,Liabilities,NetIncomeLoss,OperatingExpenses,OperatingIncomeLoss,RevenueFromContractWithCustomerExcludingAssessedTax,WeightedAverageNumberOfSharesOutstandingBasic,company,ticker,Revenues
0,2018-11-08,2018,10-Q,Q3,9.994600e+07,9.994600e+07,9.994600e+07,9.994600e+07,9.994600e+07,9.994600e+07,99946000.0,9.994600e+07,9.994600e+07,9.994600e+07,UPWK,UPWK,NaN
1,2019-03-07,2018,10-K,FY,1.018670e+08,1.018670e+08,1.018670e+08,1.018670e+08,1.018670e+08,1.018670e+08,101867000.0,1.018670e+08,1.018670e+08,1.018670e+08,UPWK,UPWK,NaN
2,2019-05-08,2019,10-Q,Q1,3.960100e+07,3.960100e+07,3.960100e+07,3.960100e+07,3.960100e+07,3.960100e+07,39601000.0,3.960100e+07,3.960100e+07,3.960100e+07,UPWK,UPWK,NaN
3,2019-08-07,2019,10-Q,Q2,8.182500e+07,8.182500e+07,8.182500e+07,8.182500e+07,8.182500e+07,8.182500e+07,81825000.0,8.182500e+07,8.182500e+07,8.182500e+07,UPWK,UPWK,NaN
4,2019-11-06,2019,10-Q,Q3,1.254340e+08,1.254340e+08,1.254340e+08,1.254340e+08,1.254340e+08,1.254340e+08,125434000.0,1.254340e+08,1.254340e+08,1.254340e+08,UPWK,UPWK,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
449,2022-05-06,2022,10-Q,Q1,9.044000e+08,9.044000e+08,9.044000e+08,9.044000e+08,9.044000e+08,9.044000e+08,NaN,9.044000e+08,9.044000e+08,9.044000e+08,AVON,AVON,9.044000e+08
450,2022-08-12,2022,10-Q,Q2,1.821400e+09,1.821400e+09,1.821400e+09,1.821400e+09,1.821400e+09,1.821400e+09,NaN,1.821400e+09,1.821400e+09,1.821400e+09,AVON,AVON,1.821400e+09
451,2022-11-10,2022,10-Q,Q3,2.589700e+09,2.589700e+09,2.589700e+09,2.589700e+09,2.589700e+09,2.589700e+09,NaN,2.589700e+09,2.589700e+09,2.589700e+09,AVON,AVON,2.589700e+09
452,2023-03-14,2022,10-K,FY,3.625200e+09,3.625200e+09,3.625200e+09,3.625200e+09,3.625200e+09,3.625200e+09,NaN,3.625200e+09,3.625200e+09,3.625200e+09,AVON,AVON,3.625200e+09


In [ ]:
# sp500_edgar_df_2 = get_all_companies_financials(sp500_ciks_dict)
# sp500_edgar_df_2 = sp500_edgar_df_2[sp500_edgar_df_2['fiscal_year'] != 2026].reset_index(drop=True)
# sp500_edgar_df_2.to_csv("../../data/edgar_sec/sp500_edgar_raw.csv", index=False)

sp500_edgar_df_2 = pd.read_csv("../../data/edgar_sec/sp500_edgar_sec_raw.csv")

sp500_edgar_df_2

,filed_date,fiscal_year,form,fiscal_period,Assets,CommonStockSharesOutstanding,IncomeTaxExpenseBenefit,Liabilities,NetIncomeLoss,OperatingIncomeLoss,RevenueFromContractWithCustomerExcludingAssessedTax,Revenues,WeightedAverageNumberOfSharesOutstandingBasic,company,ticker,GrossProfit,OperatingExpenses
0,2009-07-31,2009,10-Q,Q2,2.950000e+09,2.950000e+09,2.950000e+09,2.950000e+09,2.950000e+09,2.950000e+09,2.950000e+09,NaN,2.950000e+09,MMM,MMM,NaN,NaN
1,2009-10-30,2009,10-Q,Q3,4.463000e+09,4.463000e+09,4.463000e+09,4.463000e+09,4.463000e+09,4.463000e+09,4.463000e+09,NaN,4.463000e+09,MMM,MMM,NaN,NaN
2,2010-02-16,2009,10-K,FY,6.193000e+09,6.193000e+09,6.193000e+09,6.193000e+09,6.193000e+09,6.193000e+09,6.193000e+09,NaN,6.193000e+09,MMM,MMM,NaN,NaN
3,2010-05-05,2010,10-Q,Q1,8.030000e+08,8.030000e+08,8.030000e+08,8.030000e+08,8.030000e+08,8.030000e+08,8.030000e+08,NaN,8.030000e+08,MMM,MMM,NaN,NaN
4,2010-08-04,2010,10-Q,Q2,1.994000e+09,1.994000e+09,1.994000e+09,1.994000e+09,1.994000e+09,1.994000e+09,1.994000e+09,NaN,1.994000e+09,MMM,MMM,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29454,2025-02-13,2024,10-K,FY,8.080000e+09,8.080000e+09,8.080000e+09,8.080000e+09,8.080000e+09,NaN,NaN,8.080000e+09,8.080000e+09,ZTS,ZTS,NaN,NaN
29455,2025-05-06,2025,10-Q,Q1,2.190000e+09,2.190000e+09,2.190000e+09,2.190000e+09,2.190000e+09,NaN,NaN,2.190000e+09,2.190000e+09,ZTS,ZTS,NaN,NaN
29456,2025-08-05,2025,10-Q,Q2,4.551000e+09,4.551000e+09,4.551000e+09,4.551000e+09,4.551000e+09,NaN,NaN,4.551000e+09,4.551000e+09,ZTS,ZTS,NaN,NaN
29457,2025-11-04,2025,10-Q,Q3,6.939000e+09,6.939000e+09,6.939000e+09,6.939000e+09,6.939000e+09,NaN,NaN,6.939000e+09,6.939000e+09,ZTS,ZTS,NaN,NaN


In [13]:
print(sp500_edgar_df_2.isna().sum())

filed_date                                                 0
fiscal_year                                                0
form                                                       0
fiscal_period                                              1
Assets                                                    26
CommonStockSharesOutstanding                            6473
IncomeTaxExpenseBenefit                                  215
Liabilities                                             7806
NetIncomeLoss                                            464
OperatingIncomeLoss                                     5473
RevenueFromContractWithCustomerExcludingAssessedTax     8742
Revenues                                               15380
WeightedAverageNumberOfSharesOutstandingBasic            376
company                                                    0
ticker                                                     0
GrossProfit                                            16640
OperatingExpenses       

In [14]:
balance_sheet_cols = [col for col in concepts_to_fetch if col not in ['RevenueFromContractWithCustomerExcludingAssessedTax', 'Revenues']]
# add 'Revenue' to balance sheet cols if it exists
balance_sheet_cols.append('Revenue')
# balance_sheet_cols

In [15]:
def column_changes(df):
    df = df.copy()
    
    # if 'date' column doesn't exist, create it based on 'fiscal_period' and 'fiscal_year'
    if 'date' not in df.columns:
        fiscal_period_mapping = {
            'Q1': '03-31',
            'Q2': '06-30',
            'Q3': '09-30',
            'Q4': '12-31',
            'FY': '12-31'
        }
        
        df['date'] = df['fiscal_period'].map(fiscal_period_mapping).fillna(df['fiscal_period'])
        # uss 'fiscal_year' and 'date' to create a proper datetime column
        df['date'] = df.apply(lambda row: f"{row['fiscal_year']}-{row['date']}" if pd.notna(row['fiscal_year']) and pd.notna(row['date']) else pd.NaT, axis=1)
    
    df = df.sort_values(['ticker', 'date'])
    
    # combine all Revenue-related columns into a single 'Revenue' column
    if 'Revenues' in df.columns:
        revenue_cols = [col for col in df.columns if col != 'Revenues' and 'Revenue' in col]
        df['Revenue'] = df['Revenues']
        for col in revenue_cols:
            df['Revenue'] = df['Revenue'].combine_first(df[col])
        # drop the original Revenue columns except 'Revenue'
        df = df.drop(columns=[col for col in ['Revenues'] + revenue_cols if col != 'Revenue'], errors='ignore')
    
    # add indicator columns for whether key financial metrics are present
    df['has_gross_profit'] = df['GrossProfit'].notna().astype(int)
    df['has_operating_income'] = df['OperatingIncomeLoss'].notna().astype(int)
    df['has_operating_expenses'] = df['OperatingExpenses'].notna().astype(int)
    df['has_liabilities'] = df['Liabilities'].notna().astype(int)
    df['has_revenue'] = df['Revenue'].notna().astype(int)
    df['has_common_stock_outstanding'] = df['CommonStockSharesOutstanding'].notna().astype(int)
    df['has_weighted_avg_shares'] = df['WeightedAverageNumberOfSharesOutstandingBasic'].notna().astype(int)
    
    # ticker to 1st column
    cols = df.columns.tolist()
    cols.insert(0, cols.pop(cols.index('ticker')))
    cols.insert(1, cols.pop(cols.index('date')))
    df = df[cols]
    
    return df.reset_index(drop=True)

In [16]:
# function to add 1 period lag based on 'fiscal_period'
# def add_lagged_metrics(df, balance_sheet_cols, lag=1):
#     df = df.copy()
#     # Map for fiscal period ordering
#     period_order = {'Q1': 1, 'Q2': 2, 'Q3': 3, 'FY': 4}
#     df['fiscal_period_order'] = df['fiscal_period'].map(period_order)
    
#     df = df.sort_values(['ticker', 'fiscal_year', 'fiscal_period_order']).reset_index(drop=True)
#     for metric in balance_sheet_cols:
#         lagged_metric_name = f"{metric}_lag{lag}"
#         df[lagged_metric_name] = df.groupby('ticker')[metric].shift(lag)
#     # drop the temporary fiscal_period_order column
#     df = df.drop(columns=['fiscal_period_order'])
#     return df

In [30]:
gig_edgar_copy = gig_edgar_df_2.copy()

gig_edgar_df = column_changes(gig_edgar_copy)
# gig_edgar_lagged_df = add_lagged_metrics(gig_edgar_filled_df, balance_sheet_cols)
        
# Break into seperate 10K+20F and 10Q datasets for easier analysis
gig_edgar_10K_df = gig_edgar_df[gig_edgar_df['form'].isin(['10-K', '20-F'])].reset_index(drop=True)
# gig_edgar_10Q_df = gig_edgar_df[gig_edgar_df['form'] == '10-Q'].reset_index(drop=True)

gig_edgar_df.to_csv("../../data/edgar_sec/gig_edgar_sec_all.csv", index=False)
gig_edgar_10K_df.to_csv("../../data/edgar_sec/gig_edgar_sec_10K_20F_only.csv", index=False)
print(gig_edgar_df.isna().sum())
gig_edgar_df

ticker                                             0
date                                               0
filed_date                                         0
fiscal_year                                        0
form                                               0
fiscal_period                                      0
Assets                                             0
CommonStockSharesOutstanding                      75
GrossProfit                                      165
IncomeTaxExpenseBenefit                            0
Liabilities                                       35
NetIncomeLoss                                      0
OperatingExpenses                                338
OperatingIncomeLoss                               74
WeightedAverageNumberOfSharesOutstandingBasic      0
company                                            0
Revenue                                           18
has_gross_profit                                   0
has_operating_income                          

,ticker,date,filed_date,fiscal_year,form,fiscal_period,Assets,CommonStockSharesOutstanding,GrossProfit,IncomeTaxExpenseBenefit,...,WeightedAverageNumberOfSharesOutstandingBasic,company,Revenue,has_gross_profit,has_operating_income,has_operating_expenses,has_liabilities,has_revenue,has_common_stock_outstanding,has_weighted_avg_shares
0,AVON,2009-06-30,2009-07-30,2009,10-Q,Q2,5.237800e+09,5.237800e+09,5.237800e+09,5.237800e+09,...,5.237800e+09,AVON,5.237800e+09,1,1,0,1,1,1,1
1,AVON,2009-09-30,2009-10-29,2009,10-Q,Q3,7.882500e+09,7.882500e+09,7.882500e+09,7.882500e+09,...,7.882500e+09,AVON,7.882500e+09,1,1,0,1,1,1,1
2,AVON,2009-12-31,2010-02-25,2009,10-K,FY,9.938700e+09,9.938700e+09,9.938700e+09,9.938700e+09,...,9.938700e+09,AVON,9.938700e+09,1,1,0,1,1,1,1
3,AVON,2010-03-31,2010-04-30,2010,10-Q,Q1,2.186900e+09,2.186900e+09,2.186900e+09,2.186900e+09,...,2.186900e+09,AVON,2.186900e+09,1,1,0,1,1,1,1
4,AVON,2010-06-30,2010-07-29,2010,10-Q,Q2,4.665200e+09,4.665200e+09,4.665200e+09,4.665200e+09,...,4.665200e+09,AVON,4.665200e+09,1,1,0,1,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
449,UPWK,2024-12-31,2025-02-13,2024,10-K,FY,4.579160e+08,4.579160e+08,4.579160e+08,4.579160e+08,...,4.579160e+08,UPWK,4.579160e+08,1,1,1,1,1,1,1
450,UPWK,2025-03-31,2025-05-05,2025,10-Q,Q1,1.467440e+08,1.467440e+08,1.467440e+08,1.467440e+08,...,1.467440e+08,UPWK,1.467440e+08,1,1,1,1,1,1,1
451,UPWK,2025-06-30,2025-08-06,2025,10-Q,Q2,2.960210e+08,2.960210e+08,2.960210e+08,2.960210e+08,...,2.960210e+08,UPWK,2.960210e+08,1,1,1,1,1,1,1
452,UPWK,2025-09-30,2025-11-04,2025,10-Q,Q3,4.463890e+08,4.463890e+08,4.463890e+08,4.463890e+08,...,4.463890e+08,UPWK,4.463890e+08,1,1,1,1,1,1,1


In [28]:
sp500_edgar_copy = sp500_edgar_df_2.copy()

sp500_edgar_df = column_changes(sp500_edgar_copy)
# sp500_edgar_lagged_df = add_lagged_metrics(sp500_edgar_filled_df, balance_sheet_cols)
        
# Break into seperate 10K+20F and 10Q datasets for easier analysis
sp500_edgar_10K_20F_df = sp500_edgar_df[sp500_edgar_df['form'].isin(['10-K', '20-F'])].reset_index(drop=True)
# sp500_edgar_10Q_df = sp500_edgar_df[sp500_edgar_df['form'] == '10-Q'].reset_index(drop=True)

sp500_edgar_df.to_csv("../../data/edgar_sec/sp500_edgar_sec_all.csv", index=False)
sp500_edgar_10K_20F_df.to_csv("../../data/edgar_sec/sp500_edgar_sec_10K_20F_only.csv", index=False)
print(sp500_edgar_df.isna().sum())
sp500_edgar_df

ticker                                               0
date                                                 1
filed_date                                           0
fiscal_year                                          0
form                                                 0
fiscal_period                                        1
Assets                                              26
CommonStockSharesOutstanding                      6428
IncomeTaxExpenseBenefit                            215
Liabilities                                       7742
NetIncomeLoss                                      460
OperatingIncomeLoss                               5451
WeightedAverageNumberOfSharesOutstandingBasic      371
company                                              0
GrossProfit                                      16577
OperatingExpenses                                18754
Revenue                                           2615
has_gross_profit                                     0
has_operat

,ticker,date,filed_date,fiscal_year,form,fiscal_period,Assets,CommonStockSharesOutstanding,IncomeTaxExpenseBenefit,Liabilities,...,GrossProfit,OperatingExpenses,Revenue,has_gross_profit,has_operating_income,has_operating_expenses,has_liabilities,has_revenue,has_common_stock_outstanding,has_weighted_avg_shares
0,A,2009-12-31,2009-12-21,2009,10-K,FY,5.840000e+08,5.840000e+08,5.840000e+08,5.840000e+08,...,NaN,NaN,5.840000e+08,0,1,0,1,1,1,1
1,A,2010-03-31,2010-03-10,2010,10-Q,Q1,2.400000e+07,2.400000e+07,2.400000e+07,2.400000e+07,...,NaN,NaN,2.400000e+07,0,1,0,1,1,1,1
2,A,2010-06-30,2010-06-07,2010,10-Q,Q2,-2.300000e+07,-2.300000e+07,-2.300000e+07,-2.300000e+07,...,NaN,NaN,-2.300000e+07,0,1,0,1,1,1,1
3,A,2010-12-31,2010-12-20,2010,10-K,FY,7.950000e+08,7.950000e+08,7.950000e+08,7.950000e+08,...,NaN,NaN,7.950000e+08,0,1,0,1,1,1,1
4,A,2011-03-31,2011-03-09,2011,10-Q,Q1,9.400000e+07,9.400000e+07,9.400000e+07,9.400000e+07,...,NaN,NaN,9.400000e+07,0,1,0,1,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29454,ZTS,2024-12-31,2025-02-13,2024,10-K,FY,8.080000e+09,8.080000e+09,8.080000e+09,8.080000e+09,...,NaN,NaN,8.080000e+09,0,0,0,1,1,1,1
29455,ZTS,2025-03-31,2025-05-06,2025,10-Q,Q1,2.190000e+09,2.190000e+09,2.190000e+09,2.190000e+09,...,NaN,NaN,2.190000e+09,0,0,0,1,1,1,1
29456,ZTS,2025-06-30,2025-08-05,2025,10-Q,Q2,4.551000e+09,4.551000e+09,4.551000e+09,4.551000e+09,...,NaN,NaN,4.551000e+09,0,0,0,1,1,1,1
29457,ZTS,2025-09-30,2025-11-04,2025,10-Q,Q3,6.939000e+09,6.939000e+09,6.939000e+09,6.939000e+09,...,NaN,NaN,6.939000e+09,0,0,0,1,1,1,1


In [29]:
# all rows with fiscal year = 2026
df_2026 = sp500_edgar_df[sp500_edgar_df['fiscal_year'] == 2026]
df_2026

,ticker,date,filed_date,fiscal_year,form,fiscal_period,Assets,CommonStockSharesOutstanding,IncomeTaxExpenseBenefit,Liabilities,...,GrossProfit,OperatingExpenses,Revenue,has_gross_profit,has_operating_income,has_operating_expenses,has_liabilities,has_revenue,has_common_stock_outstanding,has_weighted_avg_shares
